Instalimi dhe importimi i librarive të nevojshme

In [ ]:
pip install pandas

In [1]:
import pandas as pd
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

Mbledhja e të dhënave, definimi i tipeve të dhënave, kualiteti i të dhënave

In [208]:
df = pd.read_csv(r'./master.csv')

print(df.dtypes)
print(df.describe())

country                object
year                    int64
sex                    object
age                    object
suicides_no             int64
population              int64
suicides/100k pop     float64
country-year           object
HDI for year          float64
 gdp_for_year ($)      object
gdp_per_capita ($)      int64
generation             object
dtype: object
               year   suicides_no    population  suicides/100k pop  \
count  27820.000000  27820.000000  2.782000e+04       27820.000000   
mean    2001.258375    242.574407  1.844794e+06          12.816097   
std        8.469055    902.047917  3.911779e+06          18.961511   
min     1985.000000      0.000000  2.780000e+02           0.000000   
25%     1995.000000      3.000000  9.749850e+04           0.920000   
50%     2002.000000     25.000000  4.301500e+05           5.990000   
75%     2008.000000    131.000000  1.486143e+06          16.620000   
max     2016.000000  22338.000000  4.380521e+07         224.970000

Riemerimi i kolonave për përdorim më të lehtë

In [ ]:
df=df.rename(columns={'sex':'gender','gdp_per_capita ($)':'gdp_per_capita',' gdp_for_year ($) ':'gdp_for_year', 'HDI for year' : 'hdi_for_year'})

Ndryshimi i dimensionalitetit

In [ ]:
df = df[['country','year','country-year', 'gender', 'age', 'suicides_no','population','suicides/100k pop','gdp_for_year','gdp_per_capita','hdi_for_year']]

Menaxhimi i vlerave null

In [ ]:
df = df.sort_values(by=['country', 'year'])

# Forward fill and backfill within each country
df['hdi_for_year'] = df.groupby('country')['hdi_for_year'].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))

# Calculate yearly mean for each country
yearly_mean = df.groupby(['country', 'year'])['hdi_for_year'].mean().reset_index()

# Interpolate mean values for each country
yearly_mean['hdi_for_year'] = yearly_mean.groupby('country')['hdi_for_year'].transform(lambda x: x.interpolate(method='linear'))

# Merge back the means into the original DataFrame
df = df.merge(yearly_mean, on=['country', 'year'], suffixes=('', '_mean'))

# Combine original HDI with the interpolated mean
df['hdi_for_year'] = df['hdi_for_year'].combine_first(df['hdi_for_year_mean'])

df = df.drop(columns=['hdi_for_year_mean'])

Mostrimi i të dhënave (10%)

In [ ]:
sampled_data = df.sample(frac=0.1)
print(sampled_data)

              country  year        country-year  gender          age  \
25758    Turkmenistan  1990    Turkmenistan1990  female  25-34 years   
2015          Austria  2004         Austria2004    male   5-14 years   
7393   Czech Republic  2014  Czech Republic2014    male  55-74 years   
24191        Suriname  2000        Suriname2000  female  35-54 years   
6982           Cyprus  2007          Cyprus2007  female  15-24 years   
...               ...   ...                 ...     ...          ...   
26239         Ukraine  2001         Ukraine2001  female  15-24 years   
4137           Belize  2015          Belize2015  female   5-14 years   
15458      Luxembourg  1997      Luxembourg1997    male  55-74 years   
13700           Japan  2013           Japan2013  female   5-14 years   
231           Albania  2008         Albania2008    male  15-24 years   

       suicides_no  population  suicides/100k pop       gdp_for_year  \
25758           16      306400               5.22      3,189,53

Validimi i vlerave duplikate <br>
Kontrollimi i kolonave specifike

In [ ]:
duplicates_check=['country','year','gender','age']
duplicates=df.duplicated(subset=duplicates_check)

if duplicates.any():
    print("Duplicates found. Dropping duplicates.")
    df = df.drop_duplicates()
    print("\nCleaned DataFrame:")
    print(df)
else:
    print("No duplicates found. DataFrame remains unchanged.")

No duplicates found. DataFrame remains unchanged.


Kontrollimi i tërë dataframe-it


In [ ]:
duplicates = df.duplicated()

if duplicates.any():
    print("Duplicates found.")
else:
    print("No duplicates found.")

No duplicates found.


Transformimi

In [ ]:
# 1. Create a new column for total suicides per year
df['total_suicides'] = df.groupby('year')['suicides_no'].transform('sum')
# 5. Create a ratio of suicides to population
df['suicides_to_population_ratio'] = df['suicides_no'] / df['population']
print(df)

          country  year    country-year  gender          age  suicides_no  \
0         Albania  1987     Albania1987    male  15-24 years           21   
1         Albania  1987     Albania1987    male   5-14 years            0   
2         Albania  1987     Albania1987  female  55-74 years            0   
3         Albania  1987     Albania1987  female   5-14 years            0   
4         Albania  1987     Albania1987  female  25-34 years            4   
...           ...   ...             ...     ...          ...          ...   
27815  Uzbekistan  2014  Uzbekistan2014    male  25-34 years          318   
27816  Uzbekistan  2014  Uzbekistan2014    male  35-54 years          519   
27817  Uzbekistan  2014  Uzbekistan2014  female   5-14 years           44   
27818  Uzbekistan  2014  Uzbekistan2014    male  15-24 years          347   
27819  Uzbekistan  2014  Uzbekistan2014  female  55-74 years           21   

       population  suicides/100k pop    gdp_for_year  gdp_per_capita  \
0  

Diskretizimi i perpjestimit të vetëvrasjeve me numrin e popullesisë dhe i gdp në kategori më të përshtatshme

In [ ]:
ratio_bins = [-1, 0, 1e-05, 2e-05, 4e-05, 6e-05, 8e-05, 1e-04]  
ratio_labels = ['None','Very Low', 'Low', 'Medium', 'High', 'Very High', 'Extreme']

df['suicides_to_population_ratio_discretize'] = pd.cut(df['suicides_to_population_ratio'], bins=ratio_bins, labels=ratio_labels)

gdp_bins = [0, 1000, 2000, 3000]
gdp_labels = ['Low', 'Medium', 'High']
df['gdp_category'] = pd.cut(df['gdp_per_capita'], bins=gdp_bins, labels=gdp_labels)

Binarizimi i kolones 'gender'

In [ ]:
df['gender_encoded'] = df['gender'].map({'male': 1, 'female': 0})

Ruajtja e transformimeve ne nje file te ri

In [ ]:

df.to_csv('cleaned_data.csv', index=False)